## __Cycle Identification using GVAX Indicator__
- GVAX (Great Volatility Auxiliar)

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.subplots as sp
import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
os.getcwd()
# Directory
#os.chdir("c:\\Users\\hp\\Downloads\\Market Data\\Market X Data")

# <span style='color: #1289eb8e;'> Data PreProcessing <span>

In [ ]:
# Class and methods for intraday data treatment
class MarketData:

    def __init__(self, filepath):
        self.filepath = filepath
        self.data = None
        self.timeframe = None

    def load_data(self):
        """Carrega os dados"""
        self.data = pd.read_csv(self.filepath, sep="\t")

    def create_datetime(self):
        """Cria coluna DATETIME inteligente"""
        if "<TIME>" in self.data.columns:
            time_col = self.data["<TIME>"]
        else:
            time_col = "00:00:00"

        self.data["DATETIME"] = pd.to_datetime(
            self.data["<DATE>"] + " " + (time_col if isinstance(time_col, pd.Series) else time_col),
            format="%Y.%m.%d %H:%M:%S"
        )

    def ninja_cut(self):
        """Remove lixo inicial se houver mudança de tempo"""
        if "<TIME>" in self.data.columns:
            first_time = self.data["<TIME>"].iloc[0]
            idx = self.data[self.data["<TIME>"] != first_time].index

            if len(idx) > 0 and idx[0] > 1:
                self.data = self.data.iloc[(idx[0]-1):].reset_index(drop=True)

    def detect_timeframe(self):
        """Detecta timeframe via mediana da diferença"""
        diff = self.data["DATETIME"].diff().dropna()
        diff_min = round(np.median(diff.dt.total_seconds() / 60))

        tfs = {
            "1440": "D1",
            "240": "H4",
            "120": "H2",
            "60": "H1",
            "30": "M30",
            "15": "M15",
            "12": "M12"
        }

        self.timeframe = tfs.get(str(diff_min), f"M{diff_min}")

    def process(self):
        """Pipeline completo"""
        self.load_data()
        self.create_datetime()
        self.ninja_cut()
        self.detect_timeframe()

        print(f"{self.timeframe} timeframe")
        return self.data

In [ ]:
# Instancia a classe
read = 4
OIL_ = MarketData(f"OILCash#_H{read}_UPDATE.csv")
OIL = OIL_.process()
ticker_O = "CRUDE OIL (WTI)"
#------------------------------------------------------------------------------------
USDJPY_ = MarketData(f"USDJPY_H4_USDJPY.csv")
USDJPY = USDJPY_.process()
ticker_Y = "USDJPY"#"DolarYen (USDJPY)"
#------------------------------------------------------------------------------------
EURUSD_ = MarketData("EURUSD#_H4_UPDATE.csv")
EURUSD = EURUSD_.process()
ticker_E = "EuroDolar (EURUSD)"
#------------------------------------------------------------------------------------
GBPUSD_ = MarketData("GBPUSD#_H4_UPDATE.csv")
GBPUSD = GBPUSD_.process()
ticker_G = "PoundDolar (GBPUSD)"
#------------------------------------------------------------------------------------
asset_d = "USDJPY"

H4 timeframe
H4 timeframe
H4 timeframe
H4 timeframe


In [ ]:
print(f"Oil: {OIL_.timeframe}", f"JPY: {USDJPY_.timeframe}",f"EUR: {EURUSD_.timeframe}",f"GBP: {GBPUSD_.timeframe}")

Oil: H4 JPY: H4 EUR: H4 GBP: H4


In [ ]:
data_name = ['OIL', 'USDJPY', 'EURUSD', 'GBPUSD']
dt = pd.DataFrame(data_name, columns=['Security'])
set_set = [OIL, USDJPY, EURUSD, GBPUSD]
for i in dt.index:
   set_set[i].columns = ["DATE", 'TIME',"OPEN",	"HIGH",	"LOW",	"CLOSE",	"TICKVOL","VOL",	"SPREAD", 'DATETIME' ]
   set_set[i] = set_set[i].set_index("DATETIME")

In [ ]:
#--------------------------------
begin = '2014-03-25'
end = '2026-09-25'
#--------------------------------

OIL = set_set[0].loc[begin:end]
USDJPY = set_set[1].loc[begin:end]
EURUSD = set_set[2].loc[begin:end]
GBPUSD = set_set[3].loc[begin:end]